# Tutorial 31 — Optimal-Design DOE (Sobol+Refine)

Companion to **§33.5** of the GMSH User Guide. Mirrors the FinStructBench case study from `graphdoe/papers/paper_jasa.tex`.

## The paper-style design

Each design row picks one **base question** *and* a configuration of **contextual factors**. The total LLM-call budget equals `n_runs` — **not** `n_questions × n_runs`. This is what the paper calls treating `base_question` as a categorical factor of cardinality $|Q|$ inside the joint design.

| Knob | This notebook | Paper |
|---|---|---|
| Base questions $|Q|$ | 30 (graph-verified, from a GMS store) | 44 |
| Contextual factors | 10 (`core` preset) | 20 (`comprehensive`) |
| Design rows = LLM calls | 60 (smoke), scale to 300 | 880 |
| Method | `sobol+refine` | `sobol+refine` |

## What this notebook walks through

1. **API demo** — call `graphdoe.design(...)` once and read off φ_p / min_dist quality
2. **Sobol vs Sobol+Refine** — same factors, same n_runs, different sampler
3. **Real graph-verified base questions** — pull from a pre-trained GMS store
4. **Paper-style design** — `question` factor + 10 contextual factors
5. **Graph baseline** — verify ground-truth integrity (cheap)
6. **LLM run** — iterate the design rows, rephrase + Claude + score (paid)
7. **Factor-effect analysis** — single-factor logistic + joint model (AIC/BIC/ROC AUC)
8. **Scaling notes** — pushing to paper settings

Run from `kg-memory/`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
import pandas as pd

from knowlytix.harness.graphdoe.question_rephraser import QuestionRephraser
from knowlytix.harness.graphdoe.analysis import DOEAnalyzer
import knowlytix.harness.graphdoe as graphdoe

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"graphdoe.design import: {graphdoe.design.__module__}")

In [ ]:
# Resolve the Anthropic API key once and set ANTHROPIC_API_KEY so that
# every downstream LLM call (litellm-via-get_llm in cell 8's
# generate_base_questions, plus the explicit get_llm() in the LLM run
# below) picks it up. The resolver checks (1) the env var, (2)
# ~/.anthropic_key, (3) ~/.config/anthropic/api-key, (4) pass(1).
import os
from knowlytix.harness.governance.api_key import (
    get_anthropic_api_key,
    AnthropicAPIKeyError,
)
try:
    os.environ.setdefault("ANTHROPIC_API_KEY", get_anthropic_api_key())
    print("ANTHROPIC_API_KEY resolved.")
except AnthropicAPIKeyError as e:
    print(f"No Anthropic API key resolved -- LLM cells will be skipped. ({e})")


## 1. Paper-style API in two lines

`graphdoe.design()` accepts a list of question strings (or any objects with `.natural_language`) plus the contextual-factor catalog selection, and returns a `DesignMatrix` whose `matrix` DataFrame has one row per LLM call and the `question` column already chosen. No outer-product expansion.

Below: 30 placeholder questions, 10 contextual factors, 60 design rows. Pure local computation — no API spend.

We use the catalog's `core` preset (10 factors, see `graphdoe/factor_catalog.yaml`) for the contextual factors.

In [ ]:
PLACEHOLDER_QUESTIONS = [f"What is the value of metric {i}?" for i in range(30)]
CORE_FACTORS = [
    "clarity", "noise", "role", "answer_format", "reasoning_cue",
    "context_scope", "entity_aliasing", "instruction_conflict",
    "complexity", "style",
]

demo_design = graphdoe.design(
    questions=PLACEHOLDER_QUESTIONS,
    factor_names=CORE_FACTORS,
    n_runs=60,
    method="sobol+refine",
    seed=42,
    verbose=False,
)
print(f"shape: {demo_design.matrix.shape}")
print(f"columns: {list(demo_design.matrix.columns)}")
print()
print(demo_design.matrix.head(3).to_string())

## 2. Sobol vs Sobol+Refine — same factors, same n_runs

Build both designs at the configuration we'll use for the real run, and read φ_p / min_dist / coverage. Pure-local — no API spend.

In [ ]:
N_RUNS_DEMO = 60   # what we'll spend in the LLM run; scale up in §8

ds_sobol = graphdoe.design(
    questions=PLACEHOLDER_QUESTIONS, factor_names=CORE_FACTORS,
    n_runs=N_RUNS_DEMO, method="sobol", seed=42,
)
ds_refine = graphdoe.design(
    questions=PLACEHOLDER_QUESTIONS, factor_names=CORE_FACTORS,
    n_runs=N_RUNS_DEMO, method="sobol+refine", seed=42, verbose=True,
)

q_s, q_r = ds_sobol.quality(), ds_refine.quality()
rows = []
for k in ("phi_p", "min_dist", "mean_nn", "mir_avg", "mir_max", "coverage", "empty_cells"):
    rows.append({
        "metric": k,
        "sobol": q_s[k],
        "sobol+refine": q_r[k],
        "ratio": q_r[k] / q_s[k] if q_s[k] not in (0, 0.0) else float('nan'),
    })
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False, float_format='%.4f'))

**What to read off** (paper-style metric set):

- `phi_p` ↓ and `min_dist` ↑ — the headline space-filling improvement (paper reports ≈2.4× and 2.6× at $n=880, d=21$).
- `mir_avg` measures level imbalance per factor; goes up when the optimizer trades grid-balance for spread.
- `coverage` is the fraction of all marginal cells (across all factors) that get used at least once.

Different `n_runs` and factor cardinalities shift the trade-off. The paper §"Mixed-Cardinality Factor Spaces" describes when refinement preserves vs. degrades coverage.

## 3. Pull real graph-verified base questions from a GMS store

Now the actual case study. We reuse the pre-trained `ch04_ground_truth` store (no training happens here — `bench.ingest()` detects the existing store and loads it).

`bridge.generate_base_questions` walks the graph and emits questions whose ground truth is derivable from triple traversal — **graph-verified, 100% correct by construction**. Picking generators that:
- yield questions on this store
- avoid the contradiction-calibration prerequisite that geometric generators require

In [ ]:
from importlib.resources import files
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig
from knowlytix.harness.testing.bridge import generate_base_questions

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    factor_group="quick_screen",  # valid catalog group; not used by the paper-style
    n_runs=N_RUNS_DEMO,            #   flow but DOEHarnessConfig validates it
    method="sobol+refine",
    seed=42,
    enable_tracing=False,         # no per-query GMS retrain
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()          # loads pre-trained store; doesn't retrain

DESIRED_BASE_Q = 30
candidate_generators = [
    "exact_retrieval", "counting", "cross_reference", "multi_hop", "boundary",
]

base_qs = generate_base_questions(
    store=store,
    generator_names=candidate_generators,
    max_per_category=10,        # generous so we can dedupe down to DESIRED_BASE_Q
    seed=42,
)

# Deduplicate by natural_language (categorical factor levels must be unique)
seen, unique_base = set(), []
for q in base_qs:
    if q.natural_language not in seen:
        seen.add(q.natural_language)
        unique_base.append(q)
    if len(unique_base) >= DESIRED_BASE_Q:
        break

print(f"Generated {len(base_qs)} base questions, kept {len(unique_base)} unique.")
for q in unique_base[:5]:
    print(f"  [{q.category}] {q.natural_language[:80]}  ->  gt={q.ground_truth}")

## 4. Build the paper-style design

Now `graphdoe.design()` with the **real** questions. The `question` column of the resulting DataFrame already picks which base question each row asks; the other 10 columns specify the contextual factor levels for that row.

In [ ]:
design = graphdoe.design(
    questions=unique_base,           # objects with .natural_language; auto-extracted
    factor_names=CORE_FACTORS,
    n_runs=N_RUNS_DEMO,
    method="sobol+refine",
    seed=42,
    verbose=False,
)

df = design.matrix
print(f"shape: {df.shape}     factors: {len(design.factors)}")
print(f"unique questions used: {df['question'].nunique()} / {len(unique_base)}")
print(f"avg reps per used question: {len(df) / df['question'].nunique():.1f}")
print()
print("design.quality():")
for k, v in design.quality().items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 5. Graph baseline (sanity)

Before paying for any LLM call, verify ground-truth integrity. The graph baseline answers each base question by traversing triples — must be 100% by construction. We only need to check the **unique** base questions (not every design row), since rephrasing leaves ground truth invariant.

In [ ]:
from knowlytix.benchmark.scorers import score_answer

n_correct = 0
for q in unique_base:
    # bridge.adapt_question wraps gms_answer_fn in a closure that ignores its
    # argument; we pass `store` to satisfy the signature `graph_answer_fn(graph)`
    graph_answer = q.graph_answer_fn(store)
    s = score_answer(graph_answer, q.ground_truth)
    if s.correct:
        n_correct += 1
    else:
        print(f"  GT MISMATCH: {q.natural_language[:60]} -> graph={graph_answer!r}  vs  gt={q.ground_truth!r}")

print(f"Graph baseline: {n_correct}/{len(unique_base)} ({n_correct/len(unique_base):.0%})")
assert n_correct == len(unique_base), "Graph baseline must be 100% — ground truth issue otherwise"

## 6. LLM evaluator — iterate the 60 design rows

Per row: pick the base question (graph-verified ground truth attached), apply the row's 10 contextual-factor levels via `QuestionRephraser`, send to Claude, score against ground truth.

Total Claude calls = `n_runs = 60`. The API key is resolved via the canonical helper (env var, then `pass show anthropic/api-key`).

Scaling: change `N_RUNS_DEMO = 300` (or `880`) and re-run from §4 — no other changes needed.

In [ ]:
from knowlytix.core.llm import get_llm
from knowlytix.harness.governance.api_key import AnthropicAPIKeyError

import os
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ANTHROPIC_API_KEY not set -- skipping LLM run.")
    print("(Set the key per the setup cell above; see")
    print(" knowlytix/harness/governance/api_key.py for resolution order.)")
    results_df = None
else:
    llm = get_llm()
    rephraser = QuestionRephraser(method="template")
    base_q_lookup = {q.natural_language: q for q in unique_base}
    with open(config.markdown_path) as f:
        document_context = f.read()

    factor_cols = [c for c in df.columns if c != "question"]
    rows_out = []
    for i, row in df.iterrows():
        base_q = base_q_lookup[row["question"]]
        factor_assignment = {c: row[c] for c in factor_cols}
        rephrased = rephraser.rephrase(base_q, factor_assignment)

        # Single-source LLM dispatch via knowlytix.core.llm.get_llm() ->
        # litellm. Same path used by cell 8's generate_base_questions, so
        # provider/key wiring lives in one place. ANTHROPIC_API_KEY env var
        # gates the call; the setup cell at the top of the notebook
        # populates it from the resolver chain.
        answer_text = llm.complete(
            messages=[{
                "role": "user",
                "content": f"<document>\n{document_context}\n</document>\n\n{rephrased.llm_prompt}",
            }],
            model="claude-sonnet-4-20250514",
            max_tokens=512,
        )
        s = score_answer(answer_text, base_q.ground_truth)

        rows_out.append({
            "row": i,
            "base_qid": base_q.qid,
            "category": base_q.category,
            "question": row["question"],
            "answer": answer_text[:400],
            "ground_truth": str(base_q.ground_truth),
            "correct": bool(s.correct),
            **factor_assignment,
        })
        if (i + 1) % 10 == 0 or i == len(df) - 1:
            n_ok = sum(r["correct"] for r in rows_out)
            print(f"  {i+1:3d}/{len(df)}  correct so far: {n_ok}/{i+1}")

    results_df = pd.DataFrame(rows_out)
    print(f"\nLLM accuracy: {results_df['correct'].sum()}/{len(results_df)} ({results_df['correct'].mean():.1%})")


## 7. Factor-effect analysis

Two complementary views — **single-factor** (which individual factors drive failures?) and **joint** (does the design as a whole explain the variation in correctness?).

The paper's headline gain is in the *joint* model: same data fits a logistic with materially better pseudo-R², AIC, BIC, and ROC AUC under Sobol+Refine vs. Sobol. `DOEAnalyzer.run_logistic` covers single-factor + interactions; `DOEAnalyzer.run_joint_model` adds the joint metrics, and `DOEAnalyzer.apply_fdr_correction` adds Benjamini–Hochberg-adjusted p-values across the family of factor tests.

> ⚠️ **Note on this smoke run.** Under exact-match scoring (used here for simplicity), Claude's paraphrased answers will often miss numeric ground truths like `0.8345` (it returns `0.83` or `approximately 0.83`). At `n_runs=60` you may see uniformly 0% LLM accuracy — that is a **scoring** artifact, not a failure of the pipeline. The design + analysis machinery still runs cleanly; logistic regression returns `NaN` p-values for a constant outcome, which is the correct behavior.
>
> For a faithful run, swap `score_answer` for the calibrated GMSJudge from Chapter 6 (`gmsh_testing_ext/notebooks/ch06_geometric_judge.ipynb`) — that's the format-tolerant scorer of record. The tutorial's purpose is to demonstrate the **paper-style design + analysis flow**, not to chase an absolute accuracy number.

In [ ]:
if results_df is None:
    print("Skipping analysis — no result.")
else:
    # Single-factor logistic
    factor_cols = [c for c in df.columns if c != "question"]
    ana = DOEAnalyzer.from_dataframe(
        results_df, factors=factor_cols + ["question"]
    )
    deviance = ana.run_logistic(metric="correct")
    cols = ["factor", "deviance", "p_value", "pseudo_r2", "significant", "n_levels"]
    print("Single-factor analysis of deviance (top 10 by p-value):")
    print(deviance.sort_values("p_value")[cols].head(10).to_string(index=False))

In [ ]:
if results_df is not None:
    # FDR-adjusted single-factor p-values (always works regardless of n_runs)
    deviance_adj = ana.apply_fdr_correction(deviance, alpha=0.05, method='fdr_bh')
    print('Single-factor with Benjamini-Hochberg FDR-adjusted p-values:')
    cols = ['factor', 'p_value', 'p_value_adj', 'significant_adj', 'pseudo_r2']
    print(deviance_adj.sort_values('p_value_adj')[cols].head(10).to_string(index=False))
    print()

    # Joint model: needs n_obs >> n_params (rule of thumb: 10x).
    # At N_RUNS_DEMO=60 with 30 questions + 10 factors, ~55 parameters need ~550 obs;
    # this fails as a singular matrix. Push N_RUNS_DEMO to >=300 (paper uses 880) for a clean fit.
    try:
        joint = ana.run_joint_model(metric='correct')
        print('Joint logistic model (all factors fitted simultaneously):')
        for k, v in joint.items():
            print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')
    except Exception as e:
        print(f'Joint model not fittable at N_RUNS_DEMO={N_RUNS_DEMO} (expected at small n): {type(e).__name__}')
        print('Scale to N_RUNS_DEMO >= 300 for a well-determined joint logistic.')

## 8. Scaling to paper settings

To replicate the paper's case study, change three knobs:

```python
DESIRED_BASE_Q = 44             # paper uses 44 graph-verified base questions
CORE_FACTORS = [...]            # paper uses 20 contextual factors (=> factor_group="comprehensive")
N_RUNS_DEMO = 880               # paper budget; ≈45 min on Claude Sonnet, ~$50–80
```

Then to compare designs, run §4–§7 twice: once with `method="sobol"`, once with `method="sobol+refine"`, and contrast the joint-model metrics from §7 (the paper reports pseudo-R² 0.752 → 0.825, ΔAIC = −90.5, ROC AUC 0.981 → 0.990 in favor of Sobol+Refine).

## See also

- **User Guide §33.5** — the prose tutorial this notebook accompanies.
- **`graphdoe/papers/paper_jasa.tex`** — full methodology, theoretical properties (Fisher information / Schur complement, fill-distance argument), and the 880-run case study.
- **`graphdoe/papers/simulation_power_study.py`** — simulation behind the +22pp statistical-power claim under known ground truth.
- **`graphdoe.analysis.DOEAnalyzer`** — single-factor logistic, Wilson CIs, odds ratios, two-way interactions, per-category decomposition. The joint-model metrics (AIC/BIC/ROC AUC/Hosmer-Lemeshow) are not yet wrapped — compute inline as in §7.